# Avito - подбор объявлений для поисковых запросов

В этой задаче для каждого запроса нужно найти 50 подходящих объявлений. Использовалась модель multilingual-e5-base, дообученную на парах "запрос - выбранное объявление". Она сравнивает тексты по смыслу, а дополнительная поправка учитывает расположение пользователя и исполнителя. Ниже находится код получения `answer.csv` из сохранённого чекпоинта `stage2/checkpoint-400`.

**Данные и подготовка текста**

В `train.parquet` содержится 497 673 положительные пары. `benchmark_items.parquet` - 189 212 объектов. Ответ строится для 2 452 запросов из `benchmark_queries.parquet`.

Для запроса использовался его текст и текстовые фильтры: они могут уточнять нужную услугу или условия. Для объявления брались заголовок, параметры и описание. Одного заголовка недостаточно: например, объявление "Наращивание ногтей" может подходить под запрос "маникюр", если эта услуга указана в других полях. Цена, рейтинг и остальные числовые признаки в итоговую оценку не входят.

Перед кодированием убираются лишние пробелы, приводится к единому виду Unicode, а пропуски в текстах заменяются пустыми строками. Из описаний удаляются ссылки и повторяющиеся строки. Из параметров в первую очередь выбираются вид и тип услуги, специальность, информация о клиентах и выезде. Объявления с одинаковым `item_id` учитываются один раз, повторяющиеся обучающие пары также удаляются. Идентификаторы сохраняются строками.

У модели ограничена длина входа, поэтому для полей выделены отдельные бюджеты: до 40 токенов на заголовок, 72 на параметры и 128 на описание. Общий лимит - 256 токенов с запасом на оформление текста.

**Как работает поиск**

E5 отдельно переводит запросы и объявления в векторы. К запросам добавляется префикс `query: `, к объявлениям - `passage: `. Векторы нормируются, после чего их скалярное произведение показывает близость текстов. Сравнение идёт со всем корпусом небольшими блоками, чтобы не хранить огромную матрицу оценок в памяти GPU.

Модель обучалась в два этапа. Сначала использовались положительные пары и другие объявления батча как негативы. При этом известные положительные объявления и их текстовые дубликаты исключались из негативов. На втором этапе добавлялись хард негативы.

К текстовому сходству добавляется мягкая поправка по географии с весом 0.035. Она рассчитывается по тому, объявления из каких локаций выбирали пользователи из данной локации в обучающей части. Редкие сочетания получают штраф, но объявления из других городов полностью не исключаются: услуга может быть удалённой или с выездом. Для этой статистики восстанавливается исходное разбиение данных; отложенные для проверки семейства объявлений исключаются из обучающей истории. После поправки выбираются 50 объявлений с наибольшей оценкой.

**Проверка качества и ограничения**

На выборке из 1 456 запросов и корпусе из 189 212 объявлений checkpoint-400 получил **Recall@50 = 0.864125**. После первой стадии результат был 0.860577, то есть вторая стадия дала около 0.355 процентного пункта прироста.

При анализе данных встретились разные формулировки одной услуги, неполные заголовки и длинные описания с повторами. Поэтому использовалась семантическая модель, добавлялись параметры и описание, а также очистку текста.

**Скачивание модели и датасета**

Модель и датасет хранятся отдельно от GitHub-репозитория. Перед запуском скачайте:

- [Модель E5, checkpoint-400](https://drive.google.com/drive/folders/1sp-8-U0qlSQgx4wnRnEQluWaSo91ugyq?usp=sharing) — поместите все файлы модели и её подпапки в `model/checkpoint-400/`. Файл `modules.json` должен находиться непосредственно в этой папке, без дополнительного вложенного каталога `checkpoint-400`.
- [Датасет](https://drive.google.com/drive/folders/1kwlU6tLn-Qb5rv-GIjX6xLgOSoXpGg5X?usp=sharing) — поместите `train.parquet`, `benchmark_items.parquet` и `benchmark_queries.parquet` непосредственно в `dataset/`.

Если файлы скачались архивом, сначала распакуйте его. Google Drive используется только для скачивания: подключать его к ноутбуку не требуется.

**Как получить ответ**

Запустить Jupyter Notebook из папки проекта и выполнить ячейки сверху вниз. Пути задаются относительно `ROOT = Path.cwd()`; если рабочая папка Jupyter другая, в этой строке нужно указать путь к проекту. Расчёт выполняется с локальными файлами, без подключения Google Drive и без зависимостей от Colab. Для повторения исходного численного режима требуется GPU **Tesla T4** и версии библиотек из `reference/provenance.json`.

Файлы нужно расположить так:

```text
project/
├── Avito_Evaluation.ipynb
├── dataset/
│   ├── train.parquet
│   ├── benchmark_items.parquet
│   └── benchmark_queries.parquet
├── artifacts/
│   ├── config.json
│   ├── split_manifest.parquet
│   └── cold_families.json
├── model/
│   └── checkpoint-400/       # исходная папка модели целиком, со всеми подпапками
├── reference/
│   ├── answer.csv           # файл, отправленный на платформу
│   └── provenance.json      # параметры и версии окружения исходного запуска
└── output/
    └── answer.csv           # создаётся при выполнении ноутбука
```

В `artifacts` нужно скопировать три одноимённых файла из исходного `runs/e5_t4_v1`. В `model/checkpoint-400` — содержимое `runs/e5_t4_v1/stage2/checkpoint-400`, сохранив структуру подпапок. Это не только веса: нужны также `modules.json`, конфигурация модели, файлы токенизатора и конфигурации модулей, например `1_Pooling/config.json`. Проще перенести весь каталог checkpoint без изменений. Не путайте `artifacts/config.json` (параметры эксперимента) и `model/checkpoint-400/config.json` (конфигурация энкодера).

В `reference` нужно положить `answer.csv` и `provenance.json` из исходного `runs/e5_t4_v1/measurements/20260928_135830_6e2de8`. Эталон используется только для сравнения: предсказания рассчитываются заново. Код проверяет версии библиотек, хеши данных, если они сохранены в конфигурации, формат ответа и побайтовое совпадение с эталоном. При несовпадении проверка остановит выполнение. Готовый файл находится в `output/answer.csv`; папка создаётся автоматически.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sentence-transformers==5.1.2", "transformers==4.57.1", "pyarrow", "pandas", "numpy", "tqdm"])

In [ ]:

from pathlib import Path
import gc, re, json, hashlib, unicodedata
import importlib.metadata as metadata
from collections import defaultdict
from types import SimpleNamespace
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# Рабочая папка Jupyter; при необходимости укажите абсолютный путь к проекту.
ROOT = Path.cwd()
DATA = ROOT / "dataset"
ARTIFACTS = ROOT / "artifacts"
MODEL_PATH = ROOT / "model" / "checkpoint-400"
REFERENCE = ROOT / "reference"
OUTPUT = ROOT / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
required = [DATA / f"{name}.parquet" for name in
            ("train", "benchmark_items", "benchmark_queries")]
required += [ARTIFACTS / name for name in
             ("config.json", "split_manifest.parquet", "cold_families.json")]
required += [MODEL_PATH / "modules.json", REFERENCE / "answer.csv", REFERENCE / "provenance.json"]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Не найдены файлы (проверьте ROOT):\n" + "\n".join(missing)
print("Папка проекта:", ROOT)
DEVICE = "cuda"
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0), "Нужна Tesla T4"
torch.set_float32_matmul_precision("high")
saved_config = json.loads((ARTIFACTS / "config.json").read_text(encoding="utf-8"))
assert saved_config.get("mode") != "smoke"
cfg = SimpleNamespace(**saved_config["config"])
cfg.encode_batch = 24
reference_provenance = json.loads((REFERENCE / "provenance.json").read_text(encoding="utf-8"))
assert reference_provenance["config"] == vars(cfg), "Изменилась конфигурация"
assert (REFERENCE / "answer.csv").is_file()
for package, version in reference_provenance["versions"].items():
    assert metadata.version(package) == version, f"Установите {package}=={version} и перезапустите среду"

def autocast_ctx():
    return torch.autocast("cuda", dtype=torch.float16)
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def stable_hash(text):
    return hashlib.blake2b(str(text).encode("utf-8"), digest_size=8).hexdigest()

In [ ]:
# Модель и данные читаются прямо из локальных папок.
assert any(MODEL_PATH.rglob("*.safetensors")) or any(MODEL_PATH.rglob("pytorch_model*.bin")), "Нет весов модели"
encoder = SentenceTransformer(str(MODEL_PATH), device=DEVICE, local_files_only=True)
encoder.max_seq_length = cfg.max_length
encoder.prompts = {"query": "query: ", "document": "passage: "}
encoder.default_prompt_name = None
encoder.eval()
print("Модель:", MODEL_PATH)

for name in ("train", "benchmark_items", "benchmark_queries"):
    path = DATA / f"{name}.parquet"
    expected = saved_config.get("data", {}).get(name, {}).get("sha256")
    if expected:
        assert file_sha256(path) == expected, f"Данные {name} отличаются от исходных"
    print(name, pq.ParquetFile(path).metadata.num_rows)


In [ ]:
SEARCH = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]
ITEM = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
        "item_category_id", "item_microcat_id", "item_price", "item_rating",
        "item_rating_reviews_count", "item_location_id", "item_latitude", "item_longitude",
        "item_is_phone_hidden", "item_is_message_forbidden"]
def clean(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    s = unicodedata.normalize("NFKC", str(text))
    return re.sub(r"\s+", " ", s).strip()
def norm(text):
    return clean(text).lower().replace("ё", "е")
def read_unique_items(path, limit=None):
    seen, parts, count = set(), [], 0
    for batch in pq.ParquetFile(path).iter_batches(batch_size=4096, columns=ITEM):
        frame = batch.to_pandas()
        frame = frame.loc[~frame.item_id.isin(seen)].drop_duplicates("item_id")
        seen.update(frame.item_id)
        for col in ("item_price", "item_latitude", "item_longitude", "item_rating", "item_rating_reviews_count"):
            frame[col] = pd.to_numeric(frame[col], errors="coerce").astype("float32")
        parts.append(frame); count += len(frame)
        if limit and count >= limit:
            break
    result = pd.concat(parts, ignore_index=True)
    return result.iloc[:limit].copy() if limit else result
def prepare_pairs(frame):
    frame = frame.copy()
    for col in ["search_query", "search_infm_params_text"]:
        frame[col] = frame[col].fillna("").map(clean)
    frame["qnorm"] = frame.search_query.map(norm)
    frame["context"] = [stable_hash(json.dumps(row, ensure_ascii=False))
                        for row in frame[SEARCH].itertuples(index=False, name=None)]
    return frame

PARAM_KEYS = ["Вид услуги","Тип услуги","Место оказания услуг","Название услуги","Услуга",
 "Стоимость","Продолжительность","Начальная цена","Тип стоимости","Где вы оказываете услуги",
 "Ваши клиенты","Кто оказывает услуги","Куда выезжаете","Чем вы занимаетесь","Специальность",
 "Опыт работы","Гарантия","Дополнительно","Время для связи, дни недели","Время для связи от",
 "Время для связи до","Работаете с юрлицами и ИП","Работа по договору","Бригада","График работы от",
 "График работы до","Готов закупить материалы"]

KEY_RE = re.compile("("+"|".join(re.escape(x) for x in sorted(PARAM_KEYS,key=len,reverse=True))+")")

PRIORITY = ["Вид услуги","Тип услуги","Название услуги","Услуга","Чем вы занимаетесь",
            "Специальность","Где вы оказываете услуги","Куда выезжаете","Ваши клиенты"]

def compact_params(text):
    text = clean(text)
    chunks = KEY_RE.split(text)
    values = defaultdict(list)
    for j in range(1,len(chunks)-1,2):
        value = clean(chunks[j+1])
        if value and value not in values[chunks[j]]:
            values[chunks[j]].append(value)
    result = "; ".join(k+": "+", ".join(values[k][:12]) for k in PRIORITY if values[k])
    return result or text[:600]

def clean_description(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    lines, seen = [], set()
    for line in re.split(r"[\r\n]+", text):
        line = clean(line)
        key = norm(line)
        if key and key not in seen:
            seen.add(key); lines.append(line)
    return " ".join(lines)

def query_text(row, filters=True):
    query = clean(row["search_query"])
    params = clean(row["search_infm_params_text"])
    if filters and params not in ("", "Вид услуги", "Тип услуги"):
        query += " | Условия: " + params
    return query

def make_document_texts(frame, tokenizer):
    result = []
    description_budget = cfg.max_length-cfg.title_tokens-cfg.params_tokens-16
    assert description_budget >= 32
    for start in tqdm(range(0,len(frame),512),desc="Подготовка текстов"):
        batch = frame.iloc[start:start+512]
        fields = [batch.item_title_raw.fillna("").map(clean).tolist(),
                  batch.item_infm_params_text.map(compact_params).tolist(),
                  batch.item_description_raw.map(clean_description).tolist()]
        decoded = []
        for values, budget in zip(fields,[cfg.title_tokens,cfg.params_tokens,description_budget]):
            ids = tokenizer(values, add_special_tokens=False, truncation=True, max_length=budget)["input_ids"]
            decoded.append(tokenizer.batch_decode(ids, skip_special_tokens=True))
        result.extend([f"{a}. Услуги: {b}. {c}" for a,b,c in zip(*decoded)])
    return result

In [ ]:
items = read_unique_items(DATA / "train.parquet")
items["family"] = [stable_hash(norm(a)+"\n"+norm(b)) for a,b in
    zip(items.item_title_raw, items.item_description_raw)]
family = items.set_index("item_id").family.to_dict()
pairs = pq.read_table(DATA / "train.parquet", columns=SEARCH+["item_id"]).to_pandas()
pairs = prepare_pairs(pairs).drop_duplicates(SEARCH+["item_id"]).reset_index(drop=True)
manifest = pd.read_parquet(ARTIFACTS / "split_manifest.parquet")
assert not manifest.duplicated(["context", "item_id"]).any()
assert len(manifest) == len(pairs), "Состав данных отличается от сохранённого разбиения"
pairs = pairs.merge(manifest[["context", "item_id", "split"]],
    on=["context", "item_id"], validate="one_to_one", how="left")
assert pairs.split.notna().all()
cold = set(json.loads((ARTIFACTS / "cold_families.json").read_text(encoding="utf-8")))
train_pairs = pairs[(pairs.split == "train") & ~pairs.item_id.map(family).isin(cold)].copy()
del pairs, manifest, family
gc.collect()
print("Пар для географии:", len(train_pairs))

In [ ]:
def encode_texts(model,texts,kind):
    with autocast_ctx():
        result = model.encode(list(texts),prompt_name=kind,batch_size=cfg.encode_batch,
            normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=len(texts)>500)
    result = np.asarray(result,dtype=np.float32)
    assert np.isfinite(result).all()
    return result

def dense_blocks(query_vectors,doc_vectors,batch=16):
    # Переносим на GPU только embeddings, не гигантскую матрицу всех пар.
    dtype = torch.float16 if DEVICE=="cuda" else torch.float32
    corpus = torch.as_tensor(np.asarray(doc_vectors),dtype=dtype,device=DEVICE)
    try:
        for start in range(0,len(query_vectors),batch):
            q = torch.as_tensor(query_vectors[start:start+batch],dtype=dtype,device=DEVICE)
            with torch.no_grad():
                scores = (q @ corpus.T).float().cpu().numpy()
            yield start,scores
    finally:
        del corpus
        if DEVICE=="cuda": torch.cuda.empty_cache()

def top_indices(scores,k):
    scores = np.asarray(scores)
    k = min(k,len(scores))
    if k<=0: return np.array([],dtype=np.int64)
    # Стабильное разрешение равенств внутри выбранной группы.
    ix = np.argpartition(-scores,k-1)[:k]
    return ix[np.lexsort((ix,-scores[ix]))]

def geography(queries, corpus, history):
    location_map = items.set_index("item_id").item_location_id.to_dict()
    transitions = history[["search_location_id","item_id"]].copy()
    transitions["destination"] = transitions.item_id.map(location_map)
    counts = transitions.groupby(["search_location_id","destination"]).size().to_dict()
    unique,inverse = np.unique(corpus.item_location_id.astype(int),return_inverse=True)
    geo = {}
    for src in queries.search_location_id.unique():
        mass = np.array([counts.get((src,dst),0)+.5+(2 if src==dst else 0) for dst in unique])
        geo[int(src)] = np.log(mass/mass.max()).clip(-12,0).astype(np.float32)
    return geo, inverse

def predict_dense(queries, corpus):
    texts = make_document_texts(corpus, encoder.tokenizer)
    geo, inverse = geography(queries, corpus, train_pairs)
    d = encode_texts(encoder, texts, "document")
    q = encode_texts(encoder, [query_text(r) for r in queries.to_dict("records")], "query")
    ids, predictions = corpus.item_id.to_numpy(), {}
    for start, scores in dense_blocks(q, d):
        for offset, score in enumerate(scores):
            row = queries.iloc[start + offset]
            adjusted = score + cfg.geo_strength * geo[int(row.search_location_id)][inverse]
            assert np.isfinite(adjusted).all()
            predictions[row.query_id] = ids[top_indices(adjusted, 50)].tolist()
    return predictions

In [ ]:
benchmark_items = read_unique_items(DATA / "benchmark_items.parquet")
benchmark_queries = pd.read_parquet(DATA / "benchmark_queries.parquet")
assert benchmark_queries.query_id.is_unique
assert benchmark_queries.query_id.map(lambda x: isinstance(x,str) and len(x)==16).all()
assert benchmark_items.item_id.is_unique and len(benchmark_items) >= 50
assert benchmark_items.item_id.str.fullmatch(r"[0-9a-f]{16}").all()
predictions = predict_dense(benchmark_queries, benchmark_items)
valid_ids = set(benchmark_items.item_id)
assert set(predictions) == set(benchmark_queries.query_id)
for q, ids in predictions.items():
    assert len(ids) == len(set(ids)) == 50, q
    assert set(ids) <= valid_ids, q
answer = pd.DataFrame({"query_id": benchmark_queries.query_id.tolist(),
    "answer": [" ".join(predictions[q]) for q in benchmark_queries.query_id]})
ANSWER_PATH = OUTPUT / "answer.csv"
answer.to_csv(ANSWER_PATH, index=False, encoding="utf-8")
check = pd.read_csv(ANSWER_PATH, dtype=str, keep_default_na=False)
assert list(check.columns) == ["query_id", "answer"] and check.equals(answer)
assert file_sha256(ANSWER_PATH) == file_sha256(REFERENCE / "answer.csv"), "CSV отличается от отправленного эталона"
print("Совпадение с эталоном подтверждено. SHA256:", file_sha256(ANSWER_PATH))
print("Готово:", ANSWER_PATH, "строк:", len(answer))
display(answer.head(3))
